<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c37-header.png" alt="Nextia Learning · Recommendation and Personalisation: Show Each Person What Matters" width="100%">

# Feedback loops and popularity bias

**[Feedback loops and popularity bias](https://learning.nextia-ai.com/courses/recommendations/m08/feedback-loops-and-popularity-bias/)** · Recommendation and Personalisation: Show Each Person What Matters · Module 8, lesson 3 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** see a recommender that learns only from its own clicks, add a small share of exploration, and measure coverage over time.

| | |
|---|---|
| **Time** | About 35 minutes |
| **Needs** | An internet connection for the setup cells (under 1 MB). No account and no key. |
| **You should know** | Coverage and popularity bias, from [Beyond accuracy](https://learning.nextia-ai.com/courses/recommendations/m03/beyond-accuracy/). |
| **Tested** | Python 3.12 with NumPy 2.5.4, pandas 3.1.1, SciPy 1.18.1 and matplotlib 3.11.2 (implicit 0.7.3 or LightGBM 4.7.0 where the lesson needs it), in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/recommendations/m08/feedback-loops-and-popularity-bias/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C37/M08-L03-feedback-loops-and-popularity-bias/feedback-loops-and-popularity-bias-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: numpy 2.5.4, pandas 3.1.1, scipy 1.18.1, and matplotlib for the pictures. It needs an internet connection and takes about a minute the first time.

You should see one line per package, with the versions above. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again.

In [ ]:
%pip install -q --disable-pip-version-check numpy==2.5.4 pandas==3.1.1 scipy==1.18.1 matplotlib==3.11.2
import warnings; warnings.filterwarnings("ignore")
from importlib.metadata import version
for name in ['numpy', 'pandas', 'scipy']:
    print(name, version(name))

## Setup: get the Larkfield data

Run the next cell. It downloads the files that this lesson needs (about 1 MB) into a folder `larkfield-recs` and checks each file's checksum. It needs an internet connection and no account, and takes about 20 seconds.

You should see `Ready:` and the number of files. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `larkfield-recs` (in Colab: the **Files** panel on the left), then run the cell again.

Where the data comes from: Larkfield is an invented shop, and its shop log is **synthetic**: our code made the products, the customers and every view, basket and order, with a model of customer behaviour. The [dataset card](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C37/data/dataset-card.md) explains how.

In [ ]:
# Setup: download the files into larkfield-recs/ and check them.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C37/"
FILES = {
    "data/online/feedback_loop.csv": "7b8ded3a30e14d30bbe2955c2a9b31a856a048778654c230fd12cfea76ab4c90",
}
HERE = Path.cwd()
# The notebook works in its own folder, marked with a file, so it never
# writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("larkfield-recs").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if not (path.exists() and sha256(path) == expected):
        local = HERE / name  # a copy next to the notebook, if there is one
        if local.exists():
            shutil.copy(local, path)
        else:
            download(LABS + name, path)
        if sha256(path) != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-recs and run again.")

os.chdir(PROJECT)  # work in the project folder, as in the lesson
print(f"Ready: {len(FILES)} files in larkfield-recs.")

## Setup: helpers

The next cell loads NumPy as `np`, pandas as `pd`, SciPy's sparse matrices as `sp` and matplotlib. It defines the course's small helpers, the same ideas as the course project:

- `products()`, `customers()` and `events()` read the data. `events()` has one row per view, basket add or order, with the column `day` (days since 1 February 2025).
- `split(period)` gives the events before the period (`train`, to learn from) and, for each customer who ordered in the period, the products they ordered (`truth`). The periods are `"validation"`, `"test"` and `"launch"`.
- `recall_at_k(list, ordered, k)`, `ndcg_at_k(list, ordered, k)` and `coverage(lists, n_products)` measure suggestion lists.
- `show(ids)` prints products with their title and subcategory.

It also defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: the data, the time split, the measures, and checks that do not show the answer.
import hashlib, json, warnings
import numpy as np
import pandas as pd
import scipy.sparse as sp
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
START = pd.Timestamp("2025-02-01")

def products():
    """One row per product (2,994)."""
    df = pd.read_csv("data/products.csv", keep_default_na=False)
    for col in ("age_restricted", "sensitive", "in_stock"):
        df[col] = df[col].astype(str).eq("True")
    return df

def customers():
    """One row per customer (40,000)."""
    return pd.read_csv("data/customers.csv")

def events():
    """One row per event: view, basket or order. day = days since 1 February 2025."""
    ev = pd.read_csv("data/events.csv.gz", parse_dates=["timestamp"])
    ev["day"] = (ev["timestamp"] - START).dt.days
    return ev

def split(period, ev=None):
    """train = the events before the period; truth = {customer: set of products ordered in the period}."""
    ev = events() if ev is None else ev
    first, last = json.load(open("data/splits.json"))[period]
    a, b = (pd.Timestamp(first) - START).days, (pd.Timestamp(last) - START).days
    train = ev[ev["day"] < a]
    target = ev[(ev["day"] >= a) & (ev["day"] <= b) & (ev["event"] == "order")]
    truth = target.groupby("customer_id")["product_id"].agg(set).to_dict()
    return train, truth

def recall_at_k(rec, ordered, k=10):
    """Share of the ordered products that are in the first k suggestions."""
    return len(set(list(rec)[:k]) & set(ordered)) / len(ordered)

def ndcg_at_k(rec, ordered, k=10):
    """DCG of the list divided by the best possible DCG (discount log2(place + 1))."""
    gains = [1.0 if p in ordered else 0.0 for p in list(rec)[:k]]
    dcg = sum(g / np.log2(i + 2) for i, g in enumerate(gains))
    ideal = sum(1 / np.log2(i + 2) for i in range(min(len(ordered), k)))
    return dcg / ideal

def coverage(lists, n_products, k=10):
    """Share of the catalogue that appears in at least one list."""
    return len({p for rec in lists for p in list(rec)[:k]}) / n_products

def rounded(x, n=3):
    """x rounded to n places (None or ... unchanged: the task is not done yet)."""
    return x if x is None or x is ... else round(float(x), n)

def show(ids, n=10):
    """Print products: place, ID, title, subcategory."""
    global _PR
    if "_PR" not in globals():
        _PR = products().set_index("product_id")
    for i, p in enumerate(list(ids)[:n], 1):
        print(f"{i:2}. {p}  {_PR.loc[p, 'title'][:52]:<52}  {_PR.loc[p, 'subcategory']}")

def fingerprint(value):
    def norm(v):
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 3)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. A toy home page that learns from its own clicks

A tiny shop that we made: 50 products, each with a hidden **appeal**. The home page shows 5 products. Each week, it shows the 5 products that got the most clicks last week. A customer looks at place k with chance 1 / k, as in the course's simulation. Run the next cell to make the shop and the loop.

In [ ]:
rng = np.random.default_rng(37)
N, SHOWN, VISITS, WEEKS = 50, 5, 500, 12             # a toy shop we made: 50 products, a home page of 5
appeal = rng.lognormal(0, 1.0, N)                    # hidden: how much people like each product
click_if_seen = appeal / (appeal + 10)               # chance of a click when a customer looks at the product
best = int(np.argmax(appeal))
print("the best product:", best, f"(click chance when seen {click_if_seen[best]:.2f});",
      "the 5 products of week 1:", list(range(SHOWN)))

def run(explore, seed=0):
    """12 weeks. Each week, show the 5 products with the most clicks last week; `explore` places are random."""
    rng = np.random.default_rng(seed)
    top = list(range(SHOWN))                         # week 1: the first 5 products of the catalogue
    history, per_visit = [], []                      # the products shown each week; clicks per visit
    for week in range(WEEKS):
        page = top[: SHOWN - explore]
        for p in rng.choice([p for p in range(N) if p not in top], explore, replace=False):
            page.insert(int(rng.integers(0, len(page) + 1)), int(p))   # a random product at a random place
        clicks = np.zeros(N)
        for _ in range(VISITS):
            for k, p in enumerate(page, 1):          # place k is looked at with chance 1 / k
                if rng.random() < 1 / k and rng.random() < click_if_seen[p]:
                    clicks[p] += 1
        history.append(page)
        per_visit.append(clicks.sum() / VISITS)
        ranked = [int(p) for p in np.argsort(-clicks, kind="stable") if clicks[p] > 0]
        top = (ranked + [p for p in top if p not in ranked])[:SHOWN]
    return history, top, per_visit

> **Predict.** With no exploration, can a product that is not on the page in week 1 ever get there?

In [ ]:
for explore in (0, 1):
    history, top, per_visit = run(explore)
    print(f"exploration {explore} of {SHOWN} places: top 5 after 12 weeks {top}; best product in it: {best in top};"
          f" clicks per visit weeks 1-4 {np.mean(per_visit[:4]):.3f}, weeks 9-12 {np.mean(per_visit[8:]):.3f}")

## 2. Your turn: coverage over time

**Coverage over time** is the number of different products shown at least once, week by week. Write `shown_so_far(history)`.

In [ ]:
def shown_so_far(history):
    """How many different products were shown at least once, week by week."""
    ...   # keep a set of the products seen so far; after each week, append its size to a list

coverage_no_explore = shown_so_far(run(0)[0])
coverage_explore = shown_so_far(run(1)[0])
print(coverage_no_explore)
print(coverage_explore)

In [ ]:
expect('coverage without exploration', coverage_no_explore, [5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5])

In [ ]:
expect('coverage with exploration', coverage_explore, [5, 6, 7, 8, 9, 10, 10, 10, 11, 12, 13, 14])

## 3. Larkfield's simulated feedback loop

The course ran the same kind of loop on its model of behaviour: 12 weeks from 6 April 2026, 6,000 customers, a home page of 10. The learner `clicks` counts clicks; the learner `ctr` uses the click rate per time shown. With exploration, 1 place of 10 (10%) or 2 places (20%) show a random product on sale. The left panel has a log scale: each step up is ten times more products.

In [ ]:
loop = pd.read_csv("data/online/feedback_loop.csv")
print("SIMULATED on the course's model of behaviour: 12 weeks, 6,000 customers")
fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
for (learner, share), g in loop.groupby(["learner", "exploration"], sort=False):
    label = f"{learner}, exploration {share:.0%}"
    ax[0].plot(g["week"], g["products_shown_so_far"], marker="o", label=label)
    ax[1].plot(g["week"], g["revenue_per_visit"], marker="o", label=label)
ax[0].set(xlabel="week", ylabel="products shown so far", yscale="log")
ax[1].set(xlabel="week", ylabel="revenue per visit (dollars)")
ax[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
summary = loop.groupby(["learner", "exploration"], sort=False).agg(
    shown_in_12_weeks=("products_shown_so_far", "last"),
    week1_products_still_in_top10=("top10_same_as_week1", "last"),
    revenue_weeks_9_12=("revenue_per_visit", lambda s: s.iloc[8:].mean()))
summary.round(2)

> **Check.** In the toy, the page without exploration showed 5 products in 12 weeks; with one random place, 14. At Larkfield, the page without exploration showed the same 10 products for 12 weeks. With 10% exploration, every one of the 2,994 products was shown, and revenue per visit in weeks 9 to 12 was lower: exploration buys coverage and information, and in this simulation it costs revenue.

In the project: `recs loop` and `recs loop --weeks`.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Batch or real time](https://learning.nextia-ai.com/courses/recommendations/m09/batch-or-real-time/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/recommendations/m08/feedback-loops-and-popularity-bias/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The Larkfield shop log is synthetic: our code made the products, the customers and everything they did, with a model of customer behaviour. No real customer is in this data.